In [ ]:
# Проверка стабильности и уникальности связки User <-> Person
if not {'User', 'Person'}.issubset(data.columns):
    print("Проверка невозможна: в data нет столбцов 'User' и/или 'Person'.")
else:
    # Уникальные пары (User, Person)
    pairs = data[['User', 'Person']].drop_duplicates()

    # 1) Стабильность: один User не должен соответствовать нескольким Person
    users_to_persons = pairs.groupby('User')['Person'].nunique()
    bad_users = users_to_persons[users_to_persons > 1]

    # 2) Уникальность в обратную сторону: один Person не должен соответствовать нескольким User
    persons_to_users = pairs.groupby('Person')['User'].nunique()
    bad_persons = persons_to_users[persons_to_users > 1]

    print(f"Уникальных пар (User, Person): {len(pairs):,}")
    print(f"User с >1 Person: {len(bad_users):,}")
    print(f"Person с >1 User: {len(bad_persons):,}")

    if len(bad_users) == 0 and len(bad_persons) == 0:
        print("Связка User ↔ Person стабильна и взаимно-уникальна (1 к 1).")
    else:
        if len(bad_users) > 0:
            print("\nПримеры User с несколькими Person:")
            display(
                pairs[pairs['User'].isin(bad_users.index)]
                .sort_values('User')
                .head(20)
            )

        if len(bad_persons) > 0:
            print("\nПримеры Person с несколькими User:")
            display(
                pairs[pairs['Person'].isin(bad_persons.index)]
                .sort_values('Person')
                .head(20)
            )

In [ ]:
data.loc[[5327762, 17093662]]

In [ ]:
# У нас есть User который однозначно идентифицирует клиента. Хранить признак его имени и фамилии нет смысла
data.drop(columns=['Person'], inplace=True)

In [ ]:
# Кросс-валидация с учетом временного ряда
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression

X = df.drop(columns=['Fraud','Timestamp'])
y = df['Fraud']

tscv = TimeSeriesSplit(n_splits=5)

scores = []

for train_idx, val_idx in tscv.split(X):
    
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)
    
    preds = model.predict_proba(X_val)[:,1]
    
    score = roc_auc_score(y_val, preds)
    scores.append(score)

print("ROC-AUC per fold:", scores)
print("Mean ROC-AUC:", sum(scores)/len(scores))

In [ ]:
df = data.copy()

# 1. подготовка
df['Timestamp'] = pd.to_datetime(df['Timestamp'])
df = df.sort_values('Timestamp').reset_index(drop=True)

# 2. historical features
df['user_txn_count'] = df.groupby('Person').cumcount()

df['merchant_txn_count'] = df.groupby('Merchant_ID').cumcount()

merchant_cum_sum = df.groupby('Merchant_ID')['Amount'].cumsum() - df['Amount']
merchant_cum_count = df.groupby('Merchant_ID').cumcount()

df['merchant_avg_amount'] = (merchant_cum_sum / merchant_cum_count).fillna(0)

df['is_first_user_for_merchant'] = (
    ~df.duplicated(subset=['Merchant_ID', 'Person'])
).astype(int)

df['merchant_unique_users'] = (
    df.groupby('Merchant_ID')['is_first_user_for_merchant']
      .cumsum()
      - df['is_first_user_for_merchant']
)

df = df.drop(columns=['is_first_user_for_merchant'])